# Problema de la mochila (Knapsack Problem)

## Definición e importancia

El **Knapsack Problem (KP)** modela una de las decisiones más fundamentales en optimización: seleccionar el subconjunto de mayor beneficio dentro de un conjunto de recursos limitados. Formalmente, de acuerdo con Kellerer et al. (2004, pp. 2–3), una instancia del KP está compuesta por un conjunto $N$ de $n$ ítems $j$, cada uno caracterizado por un beneficio $p_j$ y un peso $w_j$, junto con un valor de capacidad $c$; el objetivo es seleccionar un subconjunto de $N$ tal que el beneficio total sea máximo sin exceder $c$.

Continuando con Kellerer et al. (2004, p. 3), el KP constituye el modelo de programación entera no trivial más simple que existe, al contar con variables binarias, una única restricción y coeficientes positivos; sin embargo, la inclusión de la condición de integralidad sobre las variables $x_j$ es suficiente para ubicarlo dentro de la clase de problemas difíciles. Esta tensión entre simplicidad formal y dificultad computacional ha convertido al KP en un problema de referencia central para la disciplina, en cuyo contexto fueron introducidas técnicas fundamentales como los esquemas de aproximación, los algoritmos de reducción y la programación dinámica (Kellerer et al., 2004, pp. 4–5).

## Aplicaciones clásicas

En cuanto a sus aplicaciones, Kellerer et al. (2004, pp. 3–4) identifican tres interpretaciones clásicas del KP en contextos económicos y logísticos:

- **Problema de inversión:** un inversor selecciona proyectos con costo $w_j$ y retorno esperado $p_j$ sin exceder un presupuesto $c$.
- **Problema de carga aérea:** un despachador decide qué paquetes embarcar en una aeronave de capacidad $c$, maximizando la tarifa total.
- **Problema de corte en serrería:** un tronco de longitud $c$ se corta en piezas estándar $w_j$ con precio de venta $p_j$ para maximizar el beneficio.

Estos escenarios, aunque pertenecientes a dominios distintos, comparten la estructura esencial del KP, evidenciando su capacidad para abstraer problemas de asignación de recursos bajo restricción de capacidad.

## Aplicaciones contemporáneas

En décadas recientes, el KP ha encontrado nuevos contextos de aplicación en dominios tecnológicos de alta complejidad:

- **Computación fog/cloud:** Dai et al. (2023) emplean el método 0-1 knapsack para gestionar dinámicamente el caché de servicios en arquitecturas asistidas por la nube, donde dispositivos AIoT generan tareas sensibles al retardo y de alto consumo energético; el KP determina qué servicios almacenar en el servidor fog según la popularidad de las tareas.
- **Ingeniería eléctrica:** Oboudi et al. (2024) formulan la mejora de resiliencia sísmica en sistemas de distribución eléctrica como un KP, seleccionando medidas de refuerzo — interruptores de seccionamiento, cables subterráneos, retrofitting de subestaciones y recursos energéticos distribuidos — bajo una restricción de presupuesto.
- **Redes SDN:** Abdollahi et al. (2021) modelan el enrutamiento de flujos en centros de datos mediante un modelo knapsack, donde el ancho de banda del enlace actúa como capacidad y los flujos entrantes como ítems, asignando prioridad según el tipo de servicio y el volumen de tráfico.

Estos tres casos ilustran cómo la estructura esencial del KP — selección óptima bajo restricción de capacidad — sigue siendo un modelo vigente y productivo en la investigación aplicada contemporánea.


## Situación de Knapsack 0/1

### Enunciado

El grupo *Raccon* es un equipo de arquitectura de software dedicado a desarrollar el sistema de software *Raccon Analytics*, un analizador de tendencias de redes sociales en el cuál el usuario final puede hacer búsquedas tentativas para youtube o google trends, y el sistema devuelve búsquedas relacionadas y keywords inferidas con inteligencia artificial, junto con contenido reciente y viral de dichas redes sociales, para que el usuario tenga información valiosa para poder subir contenido público en las redes y viralizarlo.

Este grupo actualmente tiene sus componentes del sistema desplegados de manera distribuida en 4 máquinas de cómputo diferentes; cómo este grupo es muy profesional y se preocupa por la calidad del software, están tomando decisiones arquitectónicas para que su sistema cumpla con el atributo de calidad de confiabilidad, que consiste en garantizar una buena disponibilidad, resiliencia y tolerancia a fallos.
El mes que viene (Julio de 2026) comienza uno de los eventos más grandes de internet organizado por uno de los creadores de contenido más grandes, se aproxima la velada del año en su sexta edición organizada por Ibai Llanos, y con sigo muchos creadores más pequeños verán oportuno obtener visitas gracias a ello, y usarán la aplicación *Raccon Analytics* para evaluar tendencias y subir el contenido correcto.

Actualmente el sistema desplegado en las 4 computadoras soporta un estimado de 300 usuarios concurrentes, pero debido al incremento de usuarios que generará la velada, el equipo *Raccon* deberá garantizar un correcto funcionamiento con 1000 usuarios concurrentes, por lo que optan por contratar temporalmente poder computacional en los servicios de AWS, y desplegar allí réplicas de los componentes de mayor impacto en el negocio junto con balanceadores de carga y un service discovery.

El poder de cómputo a nivel de procesamiento que ofrece el contrato es más que suficiente para el equipo *Raccon* ya que el sistema mayormente solo maneja el flujo de datos y peticiones junto con API's externas, pero donde sí hay un limitante importante es en la memoria RAM.
Dicho contrato ofrece un cluster con un total de 8192MB de RAM, y dado que el equipo desplegará docker swarm junto con sus balanceadores y su DNS (Service Discovery), la memoria que queda para componentes del sistema se software son 8000MB.

Está claro que desplegar la totalidad de componentes del sistema en AWS no es viable, ya que con 1000 usuarios concurrentes, simplemente la RAM no daría abasto, por lo que por medio de pruebas de rendimiento desarrolladas por el equipo, por cada componente individual se obtuvo la cantidad de RAM en megabytes (MB) necesaria para ejecutarlo, y los resultados los anotaron en un listado de pesos denominado $W$.
Adicionalmente, hubo una discusión por parte del equipo sobre el nivel de impacto en el negocio que generaba cada uno de los componentes del sistema, y a cada uno le asignaron un número entero positivo de manera tal que un número alto indicaba alto nivel de impacto e importancia, mientras que un número bajo indicaba bajo nivel de impacto e importancia, y estos valores de impacto los anotaron en un listado de valores denominado $V$.

El equipo quiere maximizar la sumatoria de niveles de impacto de los componentes desplegados en AWS, y los componentes que no se desplieguen en AWS se mantendrán desplegados en las máquinas del equipo.

### Modelamiento matemático del problema

Definimos las listas de pesos y de valores:

$W = \{601, 7902, 6, 4520, 112, 7890, 334, 1200, 5600, 45, 6780, 230, 4400, 900, 3100, 7200, 50, 88, 3900, 7500, 2100, 6400, 500, 4200, 7700, 150, 3300, 120, 5900, 6800, 2500, 400, 7100, 950, 4800, 10, 6200, 3500, 800, 5500, 2700, 600, 7300, 1800, 4900, 7600, 320, 5400, 2200, 110\}$

$V = \{100, 567, 802, 345, 999, 120, 450, 670, 230, 890, 410, 560, 320, 770, 150, 600, 950, 430, 210, 500, 820, 390, 700, 480, 110, 660, 370, 920, 280, 530, 740, 400, 850, 190, 620, 980, 300, 550, 790, 440, 250, 690, 360, 810, 460, 130, 720, 260, 510, 930\}$

$|W| = |V| = 50$

$W_i$ es el costo en RAM del i-ésimo item

$V_i$ es el impacto estimado del i-ésimo item

#### Variables de decisión

Hay 50 variables binarias, $X_i$ es una variable binaria que indica si se elije o no el i-ésimo item

$0 \leq i < |V|$

#### Función objetivo

$Max$ $z =$ $\sum_{i=0}^{49}$ $V_i$ $\cdot$ $X_i$ unidades de impacto

#### Restricciones

Restricción binaria: $X_i \in \{0, 1\}$ $\forall$ $i \in \mathbb{Z} \cap [0, 49]$

Restricción de máximo uso de RAM: $\sum_{i=0}^{49}$ $W_i$ $\cdot$ $X_i$ $\leq$ $8000$

#### Solución con programación dinámica

In [1]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <cassert>
#include <cstring>
using namespace std;

In [2]:
const int mxW = 8000;

In [3]:
vector<bool> Knapsack(vector<int>& weig, vector<int>& val) {
  int N = weig.size();
  int dp[N][mxW + 1];
  bool take[N][mxW + 1];
  memset(dp, 0, sizeof(dp));
  memset(take, 0, sizeof(take));

  dp[0][weig[0]] = val[0];
  take[0][weig[0]] = true;

  for (int phase = 1; phase < N; ++phase) {
    int idx = phase;
    for (int w = 0; w <= mxW; ++w) {
      // Dont take
      dp[idx][w] = dp[idx - 1][w];

      // Take
      if (weig[idx] <= w && dp[idx - 1][w - weig[idx]] + val[idx] >= dp[idx][w]) {
        dp[idx][w] = dp[idx - 1][w - weig[idx]] + val[idx];
        take[idx][w] = true;
      }
    }
  }
  int best_val = *max_element(dp[N - 1], dp[N - 1] + mxW + 1);

  // Build the solution
  int current_weig = -1;
  for (int w = 0; w <= mxW; ++w) {
    if (best_val == dp[N - 1][w]) {
      current_weig = w;
      break;
    }
  }
  assert(current_weig > -1);
  int check_val = 0;
  vector<bool> X(N, false);
  for (int i = N - 1; i >= 0; --i) {
    if (take[i][current_weig]) {
      X[i] = true;
      current_weig -= weig[i];
      check_val += val[i];
    }
  }
  assert(current_weig == 0);
  assert(best_val == check_val);
  return X;
}

In [4]:
vector<int> W = {601, 7902, 6, 4520, 112, 7890, 334, 1200, 5600, 45, 6780, 230, 4400, 900, 3100, 7200, 50, 88, 3900, 7500, 2100, 6400, 500, 4200, 7700, 150, 3300, 120, 5900, 6800, 2500, 400, 7100, 950, 4800, 10, 6200, 3500, 800, 5500, 2700, 600, 7300, 1800, 4900, 7600, 320, 5400, 2200, 110};
vector<int> V = {100, 567, 802, 345, 999, 120, 450, 670, 230, 890, 410, 560, 320, 770, 150, 600, 950, 430, 210, 500, 820, 390, 700, 480, 110, 660, 370, 920, 280, 530, 740, 400, 850, 190, 620, 980, 300, 550, 790, 440, 250, 690, 360, 810, 460, 130, 720, 260, 510, 930};
vector<bool> mask = Knapsack(W, V);

int best_val = 0;
int best_weig = 0;

cout << "Los componentes a desplegar en AWS son: " << endl;
for (int i = 0; i < mask.size(); ++i) {
  if (mask[i]) {
    cout << i << ",\n"[i + 1 == mask.size()] << " \n"[i + 1 == mask.size()];
    best_val += V[i];
    best_weig += W[i];
  }
}

cout << "Mejor valor de impacto lograble con 8000MB = " << best_val << " puntos" << endl;
cout << "Memoria RAM usada = " << best_weig << "MB" << endl;

Los componentes a desplegar en AWS son: 
2, 4, 6, 7, 9, 11, 13, 16, 17, 22, 25, 27, 31, 35, 38, 41, 43, 46, 49

Mejor valor de impacto lograble con 8000MB = 14121 puntos
Memoria RAM usada = 7775MB


#### Solución óptima del equipo

Después de solucionar el problema usando programación dinámica observamos que el movimiento más inteligente es que de los 50 componentes del sistema que tiene el equipo, se desplieguen en AWS los componentes 2, 4, 6, 7, 9, 11, 13, 16, 17, 22, 25, 27, 31, 35, 38, 41, 43, 46 y 49, teniendo en cuenta que los componentes están enumerados desde 0.
Esto proporciona al equipo un valor de impacto total de 14121 puntos usando un total de 7775MB de RAM por parte de estos componentes desplegados.

# Problema de recubrimiento de conjuntos (Set Covering Problem)

## Definición e importancia

El **Set Covering Problem (SCP)** puede definirse formalmente a partir de una matriz binaria $A = (a_{ij})$ de $m$ filas y $n$ columnas, y un vector de costos enteros $c = (c_j)$. De acuerdo con Caprara et al. (2000, p. 1), se dice que una columna $j \in N$ **cubre** una fila $i \in M$ si $a_{ij} = 1$; el objetivo es encontrar un subconjunto de columnas $S \subseteq N$ de costo mínimo tal que cada fila $i \in M$ quede cubierta por al menos una columna $j \in S$.

En el contexto de la instancia evaluada, la matriz $A$ corresponde a una configuración de **50 filas y 200 columnas**, proveniente de la OR-Library de Beasley (1990), que constituye el conjunto de pruebas estándar más ampliamente utilizado en la literatura para evaluar algoritmos sobre el SCP.

La aparente simplicidad de esta formulación contrasta con su complejidad computacional. De acuerdo con Caprara et al. (2000, p. 2), el SCP es **NP-hard en sentido fuerte**, lo que lo distingue del Knapsack Problem — NP-hard en sentido débil — y lo sitúa en una categoría de mayor dificultad teórica al no admitir solución exacta en tiempo pseudo-polinomial. No obstante, como señalan Farahani et al. (2012, p. 397), los problemas de cobertura han sido siempre muy atractivos para la investigación precisamente por su aplicabilidad en problemas del mundo real, especialmente en instalaciones de servicio y emergencia, lo que ha motivado un creciente esfuerzo por desarrollar algoritmos heurísticos y exactos cada vez más eficientes para su resolución.

## Aplicaciones clásicas

El SCP fue introducido originalmente por Toregas et al. (1971) para modelar la localización de instalaciones de servicios de emergencia, específicamente estaciones de bomberos. A partir de este trabajo fundacional, Farahani et al. (2012, p. 403) documentan su extensión a:

- **Programación de tripulaciones de aerolíneas**
- **Programación integrada de vehículos y tripulaciones** en empresas de transporte
- **Localización de unidades de servicios médicos de emergencia**
- **Ubicación de sucursales bancarias**

De estas, la más representativa es la programación de tripulaciones, donde cada turno constituye una columna que cubre un subconjunto de vuelos o trayectos y el objetivo es seleccionar el conjunto de turnos de mínimo costo que garantice la cobertura completa de todas las operaciones, tal como señalan Caprara et al. (2000, p. 1) al identificarla como uno de los usos más relevantes del SCP en la industria del transporte ferroviario y de tránsito masivo.

## Aplicaciones contemporáneas

En décadas recientes, el SCP ha encontrado nuevos contextos de aplicación en dominios tecnológicos e industriales:

- **Redes sociales:** Cai et al. (2022) lo formulan para maximización de influencia en redes sociales espaciales, identificando el conjunto mínimo de usuarios que cubra la totalidad de los objetivos de una campaña publicitaria.
- **Internet of Vehicles:** Nguyen et al. (2021) lo aplican en entornos de IoV, modelando la redundancia espacio-temporal de imágenes capturadas por vehículos autónomos para minimizar datos transferidos a servidores de edge computing.
- **Transporte urbano:** Jin et al. (2022) formulan la programación de tripulaciones del metro de Beijing como un SCP extendido, validado con datos reales de tres líneas metropolitanas.
- **Seguridad industrial:** Quttineh et al. (2022) aplican un SCP biobjetivo para la ubicación óptima de cámaras térmicas en depósitos de biocombustibles, implementado en un software de soporte a decisiones actualmente en uso comercial.

## Situación de Set Covering

### Enunciado

Un parque de atracciones atiende a mucha gente a diario, sobre todo en temporadas festivas, y el uso de la maquinaría de las atracciones es muy frecuente, lo que leva a desgastes en las piñonerías, pistones y demás, por lo que el parque debe garantizar que el estado de salud de las atracciones sea correcto en todo momento y así evitar a toda costa un desafortunado accidente.

Para ello el parque de atracciones instaló paquetes de sensores en zonas críticas de cada atracción, cómo lo son los puntos que articulan las máquinas; la evaluación de cierta articulación de una atracción requiere más de un tipo de sensor, es por eso que la instalación no es por sensor individual sino por un paquete de sensores que incluyen un sensor de audio de alta impedancia y de baja impedancia, un sensor hall para campos electromagnéticos, sensor de humedad, 2 termopares para medición de temperatura y 1 foto resistencia para la luminosidad; por fines prácticos a todo un paquete lo llamaremos medidor.

Los medidores ya están instalados en las zonas importantes, sin embargo no están conectados a ninguna red, y por tanto aún no se puede consultar la información desde ningún lado, es por eso que el parque necesita instalar routers alrededor del parque que conecten todos los medidores, los medidores son para IOT por lo que se conectan a los routers inalámbricamente siempre que la distancia sea lo suficientemente pequeña y que no hayan muchos obstáculos en medio para que la señal del medidor llegue al router. Además todos los routers están conectados en una red más grande entre ellos y hacia el internet, sin importar la distancia entre router y router.

El parque aún no sabe qué tantos routers necesita, sin embargo dado el gran tamaño del parque, tiene establecidos 200 posibles lugares en donde podría instalar un router, y en total se deben conectar 50 medidores.

Cada locación de un router tiene definido un subconjunto de los 50 medidores que hay, los cuales conectaría un router en dicho lugar, y cada locación tiene implicitamente un costo de instalación del router, ya que no es lo mismo instalar un router en una habitación cómoda a hacerlo a 30 metros de altura en una posición riesgosa.

El parque quiere seleccionar estratégicamente algunas de esas 200 posibles locaciones, para instalar routers de tal forma que todos los medidores resulten conectados, y adicionalmente esto se debe lograr con el costo total de instalación de routers mínimo, el costo de instalación de un router en cada ubicación está en dólares.

### Imágen del problema

![image.png](park_routers.png)

### Modelamiento matemático del problema

Sea $C$ un listado de 200 costos, en donde $C_i$ es el costo de instalación de un router en la $i-$ésima locación.

Y sea $R$ una matriz de $50$ filas y $200$ columnas que cumple la siguiente condición:

"En la $i-$ésima fila y $j-$ésima columna (celda $R_{i,j}$) hay un $1$ si la $j-$ésima locación de router puede conectar al $i-$ésimo medidor, o en caso contrario la celda tendrá un $0$"

Para entender esta matriz prosigamos al siguiente ejemplo:

| | **Locación 1** | **Locación 2** | **Locación 3**
| :--- | :---: | :---: | :---: |
| **Medidor 1** | 0 | 1 | 1 |
| **Medidor 2** | 1 | 0 | 1 |
| **Medidor 3** | 1 | 0 | 0 |

Acá la locación 1 cubre/conecta los medidores 2 y 3.
La locación 2 cubre solo el Medidor 1.
La locación 3 cubre los medidodres 1 y 2.


#### Variables de decisión

Hay 200 variables binarias, $X_i$ es una variable binaria que indica si se instala un router en la $i-$ésima locación

$1 \leq i \leq 200$

#### Función objetivo

$Min$ $z =$ $\sum_{i=1}^{200}$ $W_i$ $\cdot$ $X_i$ dólares

#### Restricciones

Restricción binaria: $X_i \in \{0, 1\}$ $\forall$ $i \in \mathbb{Z} \cap [1, 200]$

Restricciones de cubrimiento de todos los medidores:

$\forall$ $i \in \mathbb{Z} \cap [1, 200]$, $\sum_{i=1}^{200}$ $R_{i, j}$ $\cdot$ $X_i$ $\geq$ $1$

#### Datos sobre los costos de las locaciones y el cubrimiento de medidores

El modelamiento del problema con los datos de manera completa está en el archivo SetCovering.lp, y en él se encuentra tanto los coeficientes ($C_i$) de cada variable binaria ($X_i$), cómo la información de los cubrimientos de las locaciones, es decir la información de la tabla $R$

Coeficientes ($C_i$) en SetCovering.lp:

![image.png](f_obj_set_covering.png)

Información sobre el cubrimiento de vértices (Tabla $R$):

![image.png](rest_set_covering.png)

Acá la información de cada fila de $R$ está contenida en el listado de variables que se observan en cada (cover_i), básicamente (cover_i) contiene la información de la $i-$ésima fila de $R$, si $x\_j$ aparece en el listado de $cover_i$ quiere decir que $R_{i, j} = 1$, si no aparece, entonces $R_{i, j} = 0$. En pocas palabras cada (cover_i) representa un medidor, y el listado es el listado de qué locaciones cubren a ese medidor.

#### Solución de excel:

La solución completa del problema se encuentra en el archivo set_covering_solution.xlsx adjunto en este repositorio, en dicho archivo de excel se usó solver con el método simplex LP.

![image.png](screenshot_set_covering_excel.png)

#### Análisis de la solución

La mejor decisión por parte del parque de atracciones es instalar 11 routers, de todas las 200 ubicaciones posibles para los routers, las locaciones en las cuáles se deben instalar son las 3, 10, 14, 17, 61, 70, 114, 118, 129, 165, 181. Esto da un total de 91 dólares de costo, el cuál es el mínimo alcanzable.

# Modelo de redes: AS-CAIDA 20071105

## Definición e importancia

El dataset **AS-CAIDA 20071105** fue seleccionado por su conexión directa con la infraestructura de la computación en la nube. La topología de Internet a nivel de sistemas autónomos representa la red de interconexiones que hace posible el tráfico de datos en plataformas cloud, lo que lo convierte en un modelo de alto valor para estudiar propiedades estructurales como robustez, jerarquía y conectividad a escala global. Los datos provienen de **CAIDA** (2020) y fueron consultados a través del **Network Data Repository** (Rossi y Ahmed, 2015), plataforma que consolida grafos de referencia para análisis y visualización interactiva.

Un **grafo de sistemas autónomos (AS-graph)** modela Internet a nivel inter-dominio, donde cada nodo representa una red operada por una organización independiente y cada arista una conexión entre dos de estas redes. La instancia analizada corresponde al snapshot del **5 de noviembre de 2007**, con las siguientes propiedades estructurales:

| Propiedad | Valor |
|-----------|------:|
| Nodos | 26,475 |
| Aristas | 53,381 |
| Grado promedio | 4 |
| Grado máximo | 2,628 |
| Coeficiente de clustering | 0.208 |
| $k$-core máximo | 22 |

Estos valores evidencian una topología altamente heterogénea y jerárquica. Su importancia radica en que esta estructura no es aleatoria: Faloutsos et al. (1999) demostraron que la topología de Internet obedece **leyes de potencia** estables en el tiempo, con implicaciones directas para el diseño de protocolos de enrutamiento, la ingeniería de tráfico y la selección de infraestructura en entornos cloud (CAIDA, 2020).

## Aplicaciones clásicas

Albert et al. (2000) utilizaron el grafo AS-level de Internet para demostrar que las redes de libre escala son **tolerantes a fallos aleatorios** pero **vulnerables a ataques dirigidos**. Sobre la topología de Internet específicamente, encontraron que:

- Eliminar aleatoriamente hasta el **2.5%** de los nodos no afecta el diámetro de la red.
- Eliminar ese mismo porcentaje de los **nodos más conectados** triplica el diámetro y fragmenta la red.

Este hallazgo sentó las bases del estudio de resiliencia en redes de comunicación a gran escala.

## Aplicaciones contemporáneas

Las aplicaciones actuales del modelo AS-graph se orientan hacia dos frentes:

- **Predicción de enlaces ocultos:** Wang et al. (2025) desarrollaron un método de predicción de enlaces ocultos en el grafo AS, demostrando que la topología medida sigue siendo una aproximación incompleta de la red real.
- **Detección de anomalías en enrutamiento:** Latif-Martínez et al. (2025) y Wang et al. (2026) utilizan directamente la estructura del grafo AS para detectar anomalías en el enrutamiento de Internet, superando en precisión a los métodos tradicionales basados en características de mensajes BGP.

Estos trabajos muestran cómo el modelo AS-graph ha evolucionado de herramienta de análisis estructural a base de sistemas activos de monitoreo y seguridad de infraestructura.

## Análisis del modelo de redes AS-CAIDA 20071105

A continuación se presenta una imágen de esta red

![image.png](graph_visualization.png)

Aca podemos ver visualmente el grafo que representa la red, a continuación se presenta un análisis de algunas de sus propiedades.

### Cantidad de vértices

Cada nodo en esta red representa una subred operada por una organización independiente, la cantidad de nodos en este grafo es de $26475$.

### Cantidad de aristas

Cada arista en esta red representa una conexión entre dos de las subredes de las organizaciones independientes, la cantidad de aristas en este grafo es de $53381$.

### Densidad

Esta medida en este grafo es menor a 0.000 por lo que este grafo es muy disperso, quiere decir que la relación que hay entre cantidad de nodos y cantidad de aristas, es que hay muchos nodos y pocas aristas.

### Grado máximo

El grado máximo de este grafo es $2628$, quiere decir que en todo el grafo, existe un vértice con $2628$ conexiones a otros vértices, y que no hay otro vértice con más vecinos que este.

### Grado promedio

El grado promedio de este grafo es $4.03$, es decir el promedio de los valores de los grados de todos los vértices en el grafo es $4.03$.

### Cantidad de triángulos

En los grafos muchas veces ocurre que se forman subgrafos en forma de triángulo, es decir subgrafos inducidos los cuáles son grafos completos de 3 vértices ($K_3$), en este grafo la cantidad de triángulos que hay es de $36365$.

### Cantidad máxima de triángulos

Esta métrica hace referencia a la cantidad máxima de triángulos a la que pertenece un solo nodo, en este grafo, existe un nodo que pertenece a 1295 triángulos distintos.

### Cantidad promedio de triángulos

Esta métrica hace referencia a la cantidad promedio de triángulos en los que pertenecen los vértices, es decir por cada vértice obtener la cantidad de triángulos distintos a los que pertenece, y luego promediar todos los valores para todos los vértices, en este grafo la cantidad promedio de triángulos es de $1.2$.

### Local clustering

Esta métrica dice la probabilidad de que dos vértices adyacentes a un nodo, sean adyacentes a su vez entre sí, en este grafo la probabilidad es $0.19$

### assortativity

Esta métrica al ser negativa, ($-0.52$), indica que los nodos muy conectados tienden a conectarse con nodos poco conectados, como por ejemplo en las relaciones cliente/servidor.

### Número cromático

Esta métrica indica la cantidad mínima de colores a pintar cada vértice del grafo tal que ninguna arista conecte dos vértices del mismo color, en este grafo el número cromático es 10, es decir, esto es un grafo k-partito.

### Diámetro

Esta métrica es la distancia máxima entre todos los caminos más cortos entre dos vértices, en este grafo es $8$, quiere decir que hay $2$ vértices cuyo camino más corto es de $8$ y no existe otro par de vértices cuyo _shortest path_ sea mayor a $8$.

### Distancia promedio (Mean distance)

$3.03$ es la distancia promedio en viajar entre dos pares de vértices, lo cuál es bastante bueno para ser una red tan grande.

### Conjunto independiente máximo

El conjunto independiente máximo es un conjunto de vértices tal que ninguna arista del grafo conecta dos vértices de este conjunto, y al mismo tiempo la cardinalidad de este conjunto es maximal, en este grafo la cardinalidad del conjunto independiente máximo es $6136$

### Cubrimiento mínimo de vértices

El cubrimiento mínimo de vértices es un conjunto de vértices tal que para toda arista del grafo, alguno de sus terminales es parte de este conjunto, y a su vez tiene cardinalidad maximal, este conjunto es el complemento del conjunto independiente máximo, y en este grafo la cardinalidad del cubrimiento mínimo de vértices es $20339$

### Conclusion del grafo

Algo demasiado interesante del grafo, y que destaca a simple vista son métricas como la distancia promedio y el diámetro, ya que vemos valores muy bajos para ser un grafo tan grande, esto al ser una topología de red es muy beneficioso ya que el viaje de paquetes es muy eficiente sea cuál sea el par de vértices entre los que tienen que viajar los datos.

# Bitácora de prompts
- hola, ayudame con la redaccion de una resenas de modelos de optimizacion, el primero es el knapsack problem, la resena debe tener maximo una pagina y dividirse en 4 partes, te voy a pasar unos extractos del libro de kellerer, lee la informacion y redactame solo la parte de definicion e importancia, asegurate de mencionar que es el modelo de programacion entera mas simple pero dificil de resolver por la condicion de integralidad, redactalo fluido y formal
- quedo bien la definicion, ahora sigamos con las aplicaciones historicas o clasicas, usando la misma fuente de kellerer que te pase arriba, extrae los tres ejemplos que dan el de inversion, la carga aerea y el corte en serreria, redactame esta seccion usando vinetas para que sea facil de leer y conectalo con la seccion anterior explicando que abstraen problemas de asignacion de recursos
- listo, para la tercera parte de aplicaciones contemporaneas hice una busqueda en scopus y de todos los resultados filtre estos tres articulos que son los mas relevantes: dai  sobre computacion cloud, oboudi  sobre resiliencia electrica y abdollahi  sobre redes sdn, te copio los abstracts aca abajo, leelos y redactame esta seccion explicando brevemente como cada autor aplico el knapsack problem en su campo
- perfecto, ahora junta las tres partes que redactamos definicion, aplicaciones historicas y recientes en un solo texto continuo, revisa que la transicion entre parrafos tenga sentido, que no nos pasemos de la pagina de longitud y generame la lista de referencias al final en formato apa con los autores que te fui pasando, si todo esta bien, procedemos a hacer esta misma estructura exacta de 4 pasos para el set covering problem
- listo, terminamos la mochila, ahora vamos con el set covering problem, te voy a pasar la info de caprara  para que saques la definicion, para las aplicaciones historicas usa los textos de toregas y farahani, enfocate en lo de estaciones de bomberos y tripulaciones, y para la parte reciente, hice mi busqueda en scopus y te pego aca los abstracts, lee todo eso y armame la resena completa de una pagina siguiendo la misma estructura de 4 partes que usamos antes, que quede todo bien conectado en texto continuo
- pasemos al ultimo que es el modelo de redes as-caida, aca te dejo los datos base de caida y rossi con la tabla de propiedades, mas el paper de albert  sobre la tolerancia a fallos para que armes la definicion y lo historico, para lo mas reciente filtre en scopus a wang y latif que hablan de anomalias de enrutamiento y enlaces ocultos, redactame la resena final con las 4 partes y la misma longitud de una pagina, enfocate en que quede bien clara la conexion entre la topologia de internet y la computacion en la nube
- perfecto, ahora pasame el texto de las 3 resenas a formato markdown para pegarlo directo en celdas de un jupyter notebook, arregla bien los titulos, las listas y asegurate que las formulas queden bien formateadas en latex

# Referencias

### Knapsack Problem

- Abdollahi, S., Deldari, A., Asadi, H., Montazerolghaem, A., & Mazinani, S. M. (2021). Flow-aware forwarding in SDN datacenters using a knapsack-PSO-based solution. *IEEE Transactions on Network and Service Management*, *18*(3), 2902–2914. [https://doi.org/10.1109/TNSM.2021.3064974](https://doi.org/10.1109/TNSM.2021.3064974)
- Dai, X., Xiao, Z., Jiang, H., Dustdar, S., & Liu, J. (2023). Task offloading for cloud-assisted fog computing with dynamic service caching in enterprise management systems. *IEEE Transactions on Industrial Informatics*, *19*(1), 662–672. [https://doi.org/10.1109/TII.2022.3186641](https://doi.org/10.1109/TII.2022.3186641)
- Kellerer, H., Pferschy, U., & Pisinger, D. (2004). *Knapsack problems*. Springer. [https://doi.org/10.1007/978-3-540-24777-7](https://doi.org/10.1007/978-3-540-24777-7)
- Oboudi, M. H., & Mohammadi, M. (2024). Two-stage seismic resilience enhancement of electrical distribution systems. *Reliability Engineering & System Safety*, *241*, 109635. [https://doi.org/10.1016/j.ress.2023.109635](https://doi.org/10.1016/j.ress.2023.109635)

### Set Covering Problem

- Beasley, J. E. (1990). OR-Library: Distributing test problems by electronic mail. *Journal of the Operational Research Society*, *41*(11), 1069–1072.
- Cai, T., Li, J., Mian, A., Li, R.-H., Sellis, T., & Yu, J. X. (2022). Target-aware holistic influence maximization in spatial social networks. *IEEE Transactions on Knowledge and Data Engineering*, *34*(4), 1993–2007. [https://doi.org/10.1109/TKDE.2020.3003047](https://doi.org/10.1109/TKDE.2020.3003047)
- Caprara, A., Fischetti, M., & Toth, P. (2000). Algorithms for the set covering problem. *Annals of Operations Research*, *98*(1), 353–371. [https://doi.org/10.1023/A:1019225027893](https://doi.org/10.1023/A:1019225027893)
- Farahani, R. Z., Asgari, N., Heidari, N., Hosseininia, M., & Goh, M. (2012). Covering problems in facility location: A review. *Computers & Industrial Engineering*, *62*(1), 368–407. [https://doi.org/10.1016/j.cie.2011.08.020](https://doi.org/10.1016/j.cie.2011.08.020)
- Jin, H., Chen, S., Ran, X., Liu, G., & Liu, S. (2022). Column generation-based optimum crew scheduling incorporating network representation for urban rail transit systems. *Computers & Industrial Engineering*, *169*, 108155. [https://doi.org/10.1016/j.cie.2022.108155](https://doi.org/10.1016/j.cie.2022.108155)
- Nguyen, T. D. T., Nguyen, V., Pham, V.-N., Huynh, L. N. T., Hossain, M. D., & Huh, E.-N. (2021). Modeling data redundancy and cost-aware task allocation in MEC-enabled Internet-of-Vehicles applications. *IEEE Internet of Things Journal*, *8*(3), 1687–1701. [https://doi.org/10.1109/JIOT.2020.3015534](https://doi.org/10.1109/JIOT.2020.3015534)
- Quttineh, N.-H., Olsson, P.-M., Larsson, T., & Lindell, H. (2022). An optimization approach to the design of outdoor thermal fire detection systems. *Fire Safety Journal*, *129*, 103548. [https://doi.org/10.1016/j.firesaf.2022.103548](https://doi.org/10.1016/j.firesaf.2022.103548)
- Toregas, C., Swain, R., ReVelle, C., & Bergman, L. (1971). The location of emergency services facilities. *Operations Research*, *19*(6), 1363–1373.

### Modelo de redes (AS-CAIDA)

- Albert, R., Jeong, H., & Barabási, A. L. (2000). Error and attack tolerance of complex networks. *Nature*, *406*, 378–382. [https://doi.org/10.1038/35019019](https://doi.org/10.1038/35019019)
- CAIDA. (2020). AS relationships dataset. Center for Applied Internet Data Analysis. [https://www.caida.org/catalog/datasets/as-relationships/](https://www.caida.org/catalog/datasets/as-relationships/)
- Faloutsos, M., Faloutsos, P., & Faloutsos, C. (1999). On power-law relationships of the Internet topology. En *Proceedings of the Conference on Applications, Technologies, Architectures, and Protocols for Computer Communication* (SIGCOMM '99) (pp. 251–262). Association for Computing Machinery. [https://doi.org/10.1145/316188.316229](https://doi.org/10.1145/316188.316229)
- Latif-Martínez, H., Paillissé, J., Barlet-Ros, P., & Cabellos-Aparicio, A. (2025). BGP anomaly detection using the raw internet topology. *Computer Networks*, *273*, 111753. [https://doi.org/10.1016/j.comnet.2025.111753](https://doi.org/10.1016/j.comnet.2025.111753)
- Rossi, R. A., & Ahmed, N. K. (2015). The network data repository with interactive graph analytics and visualization. En *Proceedings of the Twenty-Ninth AAAI Conference on Artificial Intelligence* (pp. 4292–4293). [https://networkrepository.com](https://networkrepository.com) [https://doi.org/10.1609/aaai.v29i1.9277](https://doi.org/10.1609/aaai.v29i1.9277)
- Wang, Z., Yuan, F., Li, R., Zhang, M., & Luo, X. (2025). Hidden AS link prediction based on random forest feature selection and GWO-XGBoost model. *Computer Networks*, *262*, 111164. [https://doi.org/10.1016/j.comnet.2025.111164](https://doi.org/10.1016/j.comnet.2025.111164)
- Wang, Z., Yuan, F., Qiu, H., Liu, Y., & Luo, X. (2026). Large-scale BGP routing anomaly detection based on graph attention auto-encoder. *IEEE Transactions on Network and Service Management*, *23*, 487–501. [https://doi.org/10.1109/TNSM.2025.3631527](https://doi.org/10.1109/TNSM.2025.3631527)
